# Tổng hợp metric đa seed từ notebook Dual-GPU

Trên Kaggle, chọn **Add Input** và thêm output của các notebook Dual-GPU cần tổng hợp (thường là hai notebook, mỗi notebook chạy hai seed). Chạy notebook này để tự tìm kết quả trong `/kaggle/input`; số seed được lấy từ dữ liệu thực tế, không cố định 4 hay 5.

Notebook đọc `seed_classification_summary.csv` / `dual_gpu_all_seeds_classification_summary.csv` và bổ sung precision, recall từ các file `*classification_report*.json` trong output của từng seed. Kết quả gồm metric từng seed, mean ± sample standard deviation, và một bảng max riêng (mỗi metric lấy giá trị cao nhất độc lập giữa các seed, kèm seed đạt max); các CSV được ghi vào `/kaggle/working/`.

Nếu dùng ngoài Kaggle, đổi `INPUT_ROOT` ở cell tiếp theo sang thư mục chứa output đã giải nén.

In [ ]:
from pathlib import Path
import json
import math
import re

import pandas as pd
from IPython.display import display

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working')
METRICS = ('Accuracy (%)', 'Macro Precision (%)', 'Macro Recall (%)', 'Macro F1 (%)')
MODEL_ORDER = ('Baseline CNN', 'CBM', 'CBM + Rule-Regularized')
BACKBONES = (
    'mobilenetv3_small', 'shufflenet_v2_x1_0', 'alexnet', 'resnet50',
    'densenet121', 'efficientnet_b0', 'swin_t', 'vit_b_16', 'vit_b_32',
)

In [ ]:
def normalize_model(value):
    text = str(value).strip().lower()
    if 'baseline' in text:
        return 'Baseline CNN'
    if 'rule' in text:
        return 'CBM + Rule-Regularized'
    if 'cbm' in text or 'concept bottleneck' in text:
        return 'CBM'
    return str(value).strip()


def model_from_report_path(path):
    parts = [part.lower() for part in path.parts]
    joined = '/'.join(parts)
    if '05_rules_model' in joined or 'rule-regularized' in joined or 'rule_regularized' in joined:
        return 'CBM + Rule-Regularized'
    if '01_baseline' in joined or 'baseline' in path.name.lower():
        return 'Baseline CNN'
    if 'cbm' in joined:
        return 'CBM'
    return None


def source_roots(input_root):
    if not input_root.exists():
        raise FileNotFoundError(f'Không tìm thấy INPUT_ROOT: {input_root}')
    roots = [path for path in input_root.iterdir() if path.is_dir()]
    return roots


def find_evidence_files(source_root):
    summary_names = {'seed_classification_summary.csv', 'dual_gpu_all_seeds_classification_summary.csv'}
    direct_summaries = [path for path in source_root.iterdir() if path.is_file() and path.name in summary_names]
    direct_seed_dirs = [path for path in source_root.glob('outputs_seed_*') if path.is_dir()]
    for seed_dir in direct_seed_dirs:
        direct_summaries.extend(path for path in seed_dir.glob('seed_classification_summary.csv') if path.is_file())
    direct_reports = []
    for seed_dir in direct_seed_dirs:
        direct_reports.extend(path for path in seed_dir.rglob('*classification_report*.json') if path.is_file())

    if direct_summaries or direct_reports:
        return direct_summaries, direct_reports

    # Kaggle may mount outputs below a generic notebook or repository folder.
    summaries = [path for name in summary_names for path in source_root.rglob(name) if path.is_file()]
    reports = [path for path in source_root.rglob('*classification_report*.json') if path.is_file()]
    return summaries, reports


def seed_from_path(path):
    for part in reversed(path.parts):
        match = re.fullmatch(r'outputs_seed_(\d+)', part)
        if match:
            return int(match.group(1))
    return None


def backbone_for(source_root, seed, evidence_path):
    config_paths = []
    if seed is not None:
        config_paths.append(evidence_path.parent)
        seed_dirs = [part for part in evidence_path.parents if part.name == f'outputs_seed_{seed}']
        if not seed_dirs:
            seed_dirs = list(source_root.rglob(f'outputs_seed_{seed}'))
        config_paths.extend(seed_dirs)
        config_paths.append(source_root / f'outputs_seed_{seed}')
    config_paths.append(source_root)
    for directory in config_paths:
        for filename in (f'params_seed_{seed}.yaml' if seed is not None else '', 'params.yaml'):
            if not filename:
                continue
            config = directory / filename
            if config.is_file():
                match = re.search(r'^\s*selected_architecture\s*:\s*[\"\']?([A-Za-z0-9_-]+)', config.read_text(encoding='utf-8'), re.MULTILINE)
                if match:
                    return match.group(1).lower()
    slug = source_root.name.lower().replace('-', '_')
    for backbone in BACKBONES:
        if backbone in slug:
            return backbone
    return 'unspecified'


def percent_value(value, already_percent=False):
    if pd.isna(value):
        return None
    number = float(value)
    if not already_percent and 0 <= number <= 1:
        number *= 100
    return number


def csv_metrics(row):
    values = {}
    columns = {str(column).strip().lower(): column for column in row.index}
    aliases = {
        'Accuracy (%)': ('accuracy (%)', 'accuracy'),
        'Macro Precision (%)': ('macro precision (%)', 'macro precision'),
        'Macro Recall (%)': ('macro recall (%)', 'macro recall'),
        'Macro F1 (%)': ('macro f1 (%)', 'macro f1', 'f1 macro (%)'),
    }
    for metric, candidates in aliases.items():
        column = next((columns[name] for name in candidates if name in columns), None)
        if column is not None:
            values[metric] = percent_value(row[column], already_percent='%' in str(column))
    return values


def report_metrics(path):
    data = json.loads(path.read_text(encoding='utf-8-sig'))
    macro = data.get('macro avg', data.get('macro_avg', {}))
    accuracy = data.get('accuracy')
    if accuracy is None:
        return {}
    return {
        'Accuracy (%)': 100 * float(accuracy),
        'Macro Precision (%)': 100 * float(macro['precision']) if macro.get('precision') is not None else None,
        'Macro Recall (%)': 100 * float(macro['recall']) if macro.get('recall') is not None else None,
        'Macro F1 (%)': 100 * float(macro['f1-score']) if macro.get('f1-score') is not None else None,
    }


records = {}

def add_record(source, backbone, seed, model, values, evidence):
    if seed is None or not model:
        return
    key = (source, backbone, int(seed), normalize_model(model))
    record = records.setdefault(key, {'metrics': {}, 'files': set()})
    record['files'].add(str(evidence))
    for metric, value in values.items():
        if value is None or pd.isna(value):
            continue
        previous = record['metrics'].get(metric)
        if previous is not None and abs(previous - float(value)) > 0.05:
            raise ValueError(f'Kết quả không khớp trong cùng input {source}, seed {seed}, {model}, {metric}: {previous} vs {value}')
        record['metrics'][metric] = float(value)


for source_root in source_roots(INPUT_ROOT):
    summaries, reports = find_evidence_files(source_root)
    for csv_path in summaries:
        frame = pd.read_csv(csv_path, encoding='utf-8-sig')
        columns = {str(column).strip().lower(): column for column in frame.columns}
        seed_column = next((columns[name] for name in ('seed', 'random seed') if name in columns), None)
        model_column = next((columns[name] for name in ('model', 'method') if name in columns), None)
        if seed_column is None or model_column is None:
            raise ValueError(f'CSV thiếu cột Seed/Model: {csv_path}')
        for _, row in frame.iterrows():
            seed = int(row[seed_column])
            add_record(source_root.name, backbone_for(source_root, seed, csv_path), seed,
                       row[model_column], csv_metrics(row), csv_path)

    for report_path in reports:
        seed = seed_from_path(report_path)
        model = model_from_report_path(report_path)
        if seed is not None and model is not None:
            add_record(source_root.name, backbone_for(source_root, seed, report_path), seed,
                       model, report_metrics(report_path), report_path)

if not records:
    raise FileNotFoundError(
        f'Không tìm thấy kết quả dual-GPU trong {INPUT_ROOT}. '
        'Hãy Add Input output notebook có seed_classification_summary.csv hoặc classification report JSON.'
    )

rows = []
for (source, backbone, seed, model), record in records.items():
    missing = [metric for metric in METRICS if metric not in record['metrics']]
    if missing:
        raise ValueError(
            f'Thiếu {missing} cho {backbone} / {model} / seed {seed} (input {source}). '
            'Đảm bảo output notebook có đầy đủ *classification_report*.json cho từng seed.'
        )
    rows.append({
        'Backbone': backbone,
        'Model': model,
        'Seed': seed,
        **record['metrics'],
        'Input': source,
        'Evidence files': '; '.join(sorted(record['files'])),
    })

per_seed = pd.DataFrame(rows)
merged_rows = []
for (backbone, model, seed), group in per_seed.groupby(['Backbone', 'Model', 'Seed'], sort=True):
    first = group.iloc[0]
    for metric in METRICS:
        values = group[metric].astype(float)
        if values.max() - values.min() > 0.05:
            raise ValueError(
                f'Trùng backbone/model/seed nhưng kết quả khác nhau: {backbone} / {model} / seed {seed} / {metric}. '
                f'Inputs: {group["Input"].tolist()}'
            )
    merged_rows.append({
        'Backbone': backbone,
        'Model': model,
        'Seed': int(seed),
        **{metric: float(group[metric].mean()) for metric in METRICS},
        'Input': ', '.join(sorted(set(group['Input']))),
    })

per_seed = pd.DataFrame(merged_rows).sort_values(['Backbone', 'Model', 'Seed']).reset_index(drop=True)
summary_rows = []
for (backbone, model), group in per_seed.groupby(['Backbone', 'Model'], sort=True):
    row = {
        'Backbone': backbone,
        'Model': model,
        'N seeds': int(group['Seed'].nunique()),
        'Seeds': ', '.join(str(seed) for seed in sorted(group['Seed'].unique())),
    }
    for metric in METRICS:
        row[f'{metric} mean'] = group[metric].mean()
        row[f'{metric} std'] = group[metric].std(ddof=1) if len(group) > 1 else 0.0
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
max_rows = []
for (backbone, model), group in per_seed.groupby(['Backbone', 'Model'], sort=True):
    row = {
        'Backbone': backbone,
        'Model': model,
        'N seeds': int(group['Seed'].nunique()),
        'Seeds': ', '.join(str(seed) for seed in sorted(group['Seed'].unique())),
    }
    for metric in METRICS:
        best_index = group[metric].idxmax()
        row[f'{metric} max'] = float(group.loc[best_index, metric])
        row[f'{metric} best seed'] = int(group.loc[best_index, 'Seed'])
    max_rows.append(row)

max_summary = pd.DataFrame(max_rows)
model_order = {name: index for index, name in enumerate(MODEL_ORDER)}
per_seed['_model_order'] = per_seed['Model'].map(model_order).fillna(len(model_order))
per_seed = per_seed.sort_values(['Backbone', '_model_order', 'Seed']).drop(columns='_model_order')
summary['_model_order'] = summary['Model'].map(model_order).fillna(len(model_order))
summary = summary.sort_values(['Backbone', '_model_order']).drop(columns='_model_order').reset_index(drop=True)
max_summary['_model_order'] = max_summary['Model'].map(model_order).fillna(len(model_order))
max_summary = max_summary.sort_values(['Backbone', '_model_order']).drop(columns='_model_order').reset_index(drop=True)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
per_seed_path = OUTPUT_ROOT / 'dual_gpu_metrics_per_seed.csv'
summary_path = OUTPUT_ROOT / 'dual_gpu_metrics_mean_std.csv'
max_summary_path = OUTPUT_ROOT / 'dual_gpu_metrics_max.csv'
per_seed.to_csv(per_seed_path, index=False, encoding='utf-8-sig')
summary.to_csv(summary_path, index=False, encoding='utf-8-sig')
max_summary.to_csv(max_summary_path, index=False, encoding='utf-8-sig')

print(f'Đã tổng hợp {len(per_seed)} kết quả seed-mô hình từ {per_seed["Input"].nunique()} input.')
display(summary)
display(max_summary)
display(per_seed)
print(f'CSV từng seed: {per_seed_path}')
print(f'CSV mean/std: {summary_path}')
print(f'CSV giá trị max theo từng metric: {max_summary_path}')